In [1]:
import os
import sys
import math
import random
import time
import subprocess
import copy
import numpy as np

def ensure(pkg):
    try:
        __import__(pkg.split("==")[0].replace("-", "_"))
        return
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["medmnist>=3.0.1", "torchvision>=0.15.0", "scikit-learn>=1.2.0", "opencv-python-headless"]:
    ensure(pkg)

import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim.swa_utils import AveragedModel, update_bn
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_recall_fscore_support, accuracy_score,
                             roc_auc_score, balanced_accuracy_score, cohen_kappa_score)
import torchvision.transforms as T
from PIL import Image
import medmnist
from medmnist import INFO

# ==============================================================================
# SECTION 1: Setup & Preprocessing
# ==============================================================================

def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # FIX: the old settings (deterministic=False, benchmark=True) let cuDNN pick
    # a different convolution algorithm on every process launch, so the SAME
    # code + SAME seed can still land on a different local minimum from run to
    # run. That alone explains "it was good before, now it's worse" without
    # anything else changing. Trade a little speed for full reproducibility:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

def _worker_init_fn(worker_id):
    # kept for completeness even though num_workers=0 everywhere right now;
    # protects you if you ever bump num_workers up.
    seed = torch.initial_seed() % (2**32)
    np.random.seed(seed)
    random.seed(seed)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device: {DEVICE}")

class CLAHE_LAB(object):
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)

    def __call__(self, img):
        img_np = np.array(img)
        lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        l_clahe = self.clahe.apply(l)
        lab_clahe = cv2.merge((l_clahe, a, b))
        rgb_clahe = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2RGB)
        return Image.fromarray(rgb_clahe)

IMG_SIZE = 192
BATCH_SIZE = 32
DATASET_FLAG = "retinamnist"

train_tfms_base = T.Compose([
    CLAHE_LAB(),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.3),
    T.RandomRotation(degrees=15),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.2),
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

train_tfms_strong = T.Compose([
    CLAHE_LAB(),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=15),
    T.ColorJitter(brightness=0.3, contrast=0.3),
    T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.3),
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

val_tfms = T.Compose([
    CLAHE_LAB(),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

info = INFO[DATASET_FLAG]
Task = getattr(medmnist, info["python_class"])
n_classes = len(info["label"])

class ClassAwareTransformDataset(Dataset):
    def __init__(self, base_dataset, base_transform, strong_transform, minority_classes, strong_p=0.8):
        self.base = base_dataset
        self.base_transform = base_transform
        self.strong_transform = strong_transform
        self.minority = set(int(c) for c in minority_classes)
        self.strong_p = float(strong_p)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, target = self.base[idx]
        if not isinstance(img, Image.Image):
            img = Image.fromarray(img)
        y = int(np.array(target).squeeze())

        if (y in self.minority) and (np.random.rand() < self.strong_p):
            img = self.strong_transform(img)
        else:
            img = self.base_transform(img)
        return img, torch.tensor(y, dtype=torch.long)

def get_dataloaders():
    train_set_base = Task(split="train", download=True, as_rgb=True, size=224)
    val_set = Task(split="val", transform=val_tfms, download=True, as_rgb=True, size=224)
    test_set = Task(split="test", transform=val_tfms, download=True, as_rgb=True, size=224)

    labels_train = np.array(train_set_base.labels).squeeze().astype(int)
    class_counts = np.bincount(labels_train, minlength=n_classes)
    median_count = np.median(class_counts)
    minority_classes = [i for i, c in enumerate(class_counts) if c < median_count]

    train_set = ClassAwareTransformDataset(
        base_dataset=train_set_base,
        base_transform=train_tfms_base,
        strong_transform=train_tfms_strong,
        minority_classes=minority_classes,
        strong_p=0.6
    )
    # FIX: give the shuffling DataLoader its own seeded generator. Without this,
    # the batch order it draws from the *global* torch RNG can differ between
    # runs depending on exactly how much other code (augmentations, model init,
    # etc.) has already consumed from that same global RNG stream beforehand.
    g = torch.Generator()
    g.manual_seed(42)
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
                               pin_memory=True, drop_last=False, generator=g, worker_init_fn=_worker_init_fn)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True, drop_last=False)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True, drop_last=False)

    return train_loader, val_loader, test_loader, class_counts, labels_train

# ==============================================================================
# SECTION 2: Model architectures
# ==============================================================================

class SEBlock(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        hidden = max(8, ch // r)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(ch, hidden, 1, bias=True), nn.SiLU(),
            nn.Conv2d(hidden, ch, 1, bias=True), nn.Sigmoid()
        )
    def forward(self, x):
        w = self.fc(self.pool(x))
        return x * w

class InvertedResidualSE(nn.Module):
    def __init__(self, in_ch, out_ch, expand_ratio=3, k=3, s=1, use_se=True):
        super().__init__()
        mid = int(in_ch * expand_ratio)
        self.use_res = (s == 1 and in_ch == out_ch)
        self.pw1 = nn.Sequential(nn.Conv2d(in_ch, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.SiLU())
        self.dw = nn.Sequential(nn.Conv2d(mid, mid, k, stride=s, padding=k//2, groups=mid, bias=False), nn.BatchNorm2d(mid), nn.SiLU())
        self.se = SEBlock(mid) if use_se else nn.Identity()
        self.pw2 = nn.Sequential(nn.Conv2d(mid, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch))
    def forward(self, x):
        y = self.pw1(x)
        y = self.dw(y)
        y = self.se(y)
        y = self.pw2(y)
        return x + y if self.use_res else y

class FeatureExtractorDW(nn.Module):
    def __init__(self, in_ch=3, base=32, downsample=True):
        super().__init__()
        s2 = 2 if downsample else 1
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, base, kernel_size=3, stride=s2, padding=1, bias=False),
            nn.BatchNorm2d(base), nn.SiLU()
        )
        self.a1 = InvertedResidualSE(base, base, expand_ratio=4, k=5, s=1, use_se=True)
        self.a2 = InvertedResidualSE(base, base, expand_ratio=4, k=3, s=1, use_se=True)
        self.ds1 = InvertedResidualSE(base, base*2, expand_ratio=4, k=5, s=2, use_se=True)
        self.b1 = InvertedResidualSE(base*2, base*2, expand_ratio=4, k=3, s=1, use_se=True)
        self.b2 = InvertedResidualSE(base*2, base*2, expand_ratio=4, k=5, s=1, use_se=True)
        self.ds2 = InvertedResidualSE(base*2, base*4, expand_ratio=4, k=5, s=2, use_se=True)
        self.c1 = InvertedResidualSE(base*4, base*4, expand_ratio=4, k=3, s=1, use_se=True)
        self.c2 = InvertedResidualSE(base*4, base*4, expand_ratio=4, k=5, s=1, use_se=True)
        self.refine = nn.Sequential(
            nn.Conv2d(base*4, base*4, kernel_size=3, padding=1, groups=base*4, bias=False),
            nn.BatchNorm2d(base*4), nn.SiLU()
        )
        self.out_ch = base*4

    def forward(self, x):
        x = self.stem(x)
        x = self.a1(x); x = self.a2(x)
        x = self.ds1(x)
        x = self.b1(x); x = self.b2(x)
        x = self.ds2(x)
        x = self.c1(x); x = self.c2(x)
        x = self.refine(x)
        return x

class GhostModule(nn.Module):
    def __init__(self, in_ch, out_ch, ratio=2, kernel_size=1, dw_kernel=3, s=1, act=True):
        super().__init__()
        init_ch = math.ceil(out_ch / ratio)
        new_ch = init_ch * (ratio - 1)
        p = kernel_size // 2
        self.primary = nn.Sequential(
            nn.Conv2d(in_ch, init_ch, kernel_size, stride=s, padding=p, bias=False),
            nn.BatchNorm2d(init_ch),
            nn.SiLU() if act else nn.Identity()
        )
        self.cheap = nn.Sequential(
            nn.Conv2d(init_ch, new_ch, dw_kernel, stride=1, padding=dw_kernel//2, groups=init_ch, bias=False),
            nn.BatchNorm2d(new_ch),
            nn.SiLU() if act else nn.Identity()
        )
        self.out_ch = out_ch
    def forward(self, x):
        y = self.primary(x); z = self.cheap(y)
        out = torch.cat([y, z], dim=1)
        return out[:, :self.out_ch, :, :]

class AdaptiveSelfAttention2d(nn.Module):
    def __init__(self, channels, max_heads=8, attn_dropout=0.1, proj_dropout=0.1, local_kernel=3, topk=3):
        super().__init__()
        self.channels = channels
        h = min(int(max_heads), channels)
        while h > 1 and (channels % h != 0): h -= 1
        self.heads = h
        self.dim_head = channels // h
        self.topk = int(topk)
        self.qkv = nn.Conv2d(channels, channels * 3, kernel_size=1, bias=True)
        self.q_norm = nn.LayerNorm(self.dim_head)
        self.k_norm = nn.LayerNorm(self.dim_head)
        self.talking_in = nn.Parameter(torch.eye(self.heads))
        self.talking_out = nn.Parameter(torch.eye(self.heads))
        self.tau = nn.Parameter(torch.ones(self.heads) * (self.dim_head ** -0.5))
        self.pos = nn.Conv2d(channels, channels, kernel_size=3, padding=1, groups=channels, bias=False)
        self.local = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=local_kernel, padding=local_kernel//2, groups=channels, bias=False),
            nn.Conv2d(channels, channels, kernel_size=1, bias=False),
            nn.SiLU()
        )
        self.alpha = nn.Parameter(torch.tensor(0.25))
        self.gamma = nn.Parameter(torch.tensor(0.5))
        self.attn_drop = nn.Dropout(attn_dropout)
        self.proj = nn.Conv2d(channels, channels, kernel_size=1, bias=False)
        self.proj_drop = nn.Dropout(proj_dropout)
        self.head_gate = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(channels, self.heads, kernel_size=1, bias=True))

    def forward(self, x):
        B, C, H, W = x.shape
        N = H * W
        x_pos = x + self.pos(x)
        qkv = self.qkv(x_pos)
        q, k, v = torch.chunk(qkv, 3, dim=1)
        def reshape(t): return t.view(B, self.heads, self.dim_head, H, W).flatten(-2).transpose(2, 3)
        q = reshape(q); k = reshape(k); v = reshape(v)
        q = self.q_norm(q.reshape(B * self.heads, N, self.dim_head)).reshape(B, self.heads, N, self.dim_head)
        k = self.k_norm(k.reshape(B * self.heads, N, self.dim_head)).reshape(B, self.heads, N, self.dim_head)
        attn = torch.einsum('b h n d, b h m d -> b h n m', q, k)
        attn = attn * self.tau.view(1, self.heads, 1, 1)
        attn = torch.einsum('hH, b h n m -> b H n m', self.talking_in, attn)
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        attn = torch.einsum('b h n m, hH -> b H n m', attn, self.talking_out)
        g_logits = self.head_gate(x).view(B, self.heads)
        g_probs = g_logits.softmax(dim=-1)
        if 1 <= self.topk < self.heads:
            topk_scores, topk_idx = torch.topk(g_probs, self.topk, dim=1)
            mask = torch.zeros(B, self.heads, device=x.device, dtype=attn.dtype)
            mask.scatter_(1, topk_idx, 1.0)
            head_weight = (mask * g_probs).view(B, self.heads, 1, 1)
        else:
            head_weight = g_probs.view(B, self.heads, 1, 1)
        attn = attn * head_weight
        out = torch.einsum('b h n m, b h m d -> b h n d', attn, v)
        out = out.transpose(2, 3).contiguous().view(B, C, H, W)
        out = self.proj_drop(self.proj(out))
        out = x + self.gamma * out + self.alpha * self.local(x)
        return out

class ABlock(nn.Module):
    def __init__(self, in_ch, base=32):
        super().__init__()
        self.rep_stage = FeatureExtractorDW(in_ch=in_ch, base=base, downsample=False)
        mid = self.rep_stage.out_ch
        self.adapt_attn = AdaptiveSelfAttention2d(
            mid, max_heads=8,
            attn_dropout=0.1,
            proj_dropout=0.1,
            topk=4
        )
        self.ghost = GhostModule(mid, mid, ratio=2, kernel_size=1, dw_kernel=3, s=1, act=True)
        self.proj = nn.Identity() if in_ch == mid else nn.Conv2d(in_ch, mid, 1, bias=False)
        self.bn = nn.BatchNorm2d(mid)
        self.act = nn.SiLU()
        self.out_ch = mid
    def forward(self, x):
        identity = self.proj(x)
        y = self.rep_stage(x)
        y = self.adapt_attn(y)
        y = self.ghost(y)
        if identity.shape[-2:] != y.shape[-2:]:
            identity = F.adaptive_avg_pool2d(identity, y.shape[-2:])
        return self.act(self.bn(y + identity))

class PatchEmbed(nn.Module):
    def __init__(self, in_ch, embed_dim, patch_size=2):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, embed_dim, kernel_size=patch_size, stride=patch_size, bias=False)
        self.norm = nn.LayerNorm(embed_dim)
    def forward(self, x):
        x = self.proj(x)
        B, D, Hp, Wp = x.shape
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)
        h = torch.linspace(0, 1, steps=Hp, device=x.device).repeat_interleave(Wp)
        w = torch.linspace(0, 1, steps=Wp, device=x.device).repeat(Hp)
        return x, h, w, Hp, Wp

def pos2d_sincos(Hp, Wp, dim, device):
    assert dim % 2 == 0
    dim_half = dim // 2
    def pe_1d(L, C):
        pos = torch.arange(L, device=device).unsqueeze(1)
        div = torch.exp(torch.arange(0, C, 2, device=device) * (-math.log(10000.0) / C))
        pe = torch.zeros(L, C, device=device)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return pe
    py = pe_1d(Hp, dim_half)
    px = pe_1d(Wp, dim_half)
    pe = torch.zeros(Hp, Wp, dim, device=device)
    for i in range(Hp):
        for j in range(Wp):
            pe[i, j, :dim_half] = py[i]
            pe[i, j, dim_half:] = px[j]
    return pe.view(1, Hp*Wp, dim)

class LinearAttention(nn.Module):
    def __init__(self, dim, heads=4, dim_head=32, out_scale=0.5, eps=1e-6):
        super().__init__()
        self.heads = heads
        self.out_scale = out_scale
        self.eps = eps
        inner = heads * dim_head
        self.to_q = nn.Linear(dim, inner, bias=False)
        self.to_k = nn.Linear(dim, inner, bias=False)
        self.to_v = nn.Linear(dim, inner, bias=False)
        self.to_out = nn.Linear(inner, dim)
        self.scale = dim_head ** -0.5
    @staticmethod
    def phi(x): return F.elu(x) + 1.0
    def forward(self, x):
        B, N, D = x.shape
        H = self.heads
        def shape(proj): return proj.view(B, N, H, -1).transpose(1, 2)
        q = self.phi(shape(self.to_q(x))) * self.scale
        k = self.phi(shape(self.to_k(x)))
        v = shape(self.to_v(x))
        kv = k.transpose(-2, -1) @ v
        z = 1.0 / (q @ k.sum(dim=-2, keepdim=True).transpose(-2, -1) + self.eps)
        out = (q @ kv) * z
        out = out.transpose(1, 2).contiguous().view(B, N, -1)
        return self.to_out(out) * self.out_scale

class PlainFFN(nn.Module):
    def __init__(self, dim, hidden_mult=2, dropout=0.1):
        super().__init__()
        hidden = dim * hidden_mult
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, dim), nn.Dropout(dropout)
        )
    def forward(self, x): return self.net(x)

class LiteBlock(nn.Module):
    def __init__(self, dim, heads=4, dim_head=32, hidden_mult=2, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = LinearAttention(dim, heads=heads, dim_head=dim_head, out_scale=0.5)
        self.norm2 = nn.LayerNorm(dim)
        self.ff = PlainFFN(dim, hidden_mult=hidden_mult, dropout=dropout)
    def forward(self, x, h_coord, w_coord):
        x = x + self.attn(self.norm1(x))
        x = x + self.ff(self.norm2(x))
        return x

class ImprovedRetinaNet(nn.Module):
    def __init__(self, in_ch=3, num_classes=5, base=48, embed_dim=192, patch=2,
                 heads=8, dim_head=32, depth=3, hidden_mult=2, dropout=0.2):
        super().__init__()
        self.stage1 = FeatureExtractorDW(in_ch=in_ch, base=base, downsample=True)
        self.drop2d = nn.Dropout2d(p=0.1)

        self.ablock = ABlock(in_ch=self.stage1.out_ch, base=base)
        ch = self.ablock.out_ch
        self.patch = PatchEmbed(ch, embed_dim, patch_size=patch)
        self.blocks = nn.ModuleList([
            LiteBlock(embed_dim, heads=heads, dim_head=dim_head, hidden_mult=hidden_mult, dropout=dropout)
            for _ in range(depth)
        ])
        self.final_norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(embed_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.SiLU(),
            nn.Dropout(p=0.3),
            nn.Linear(embed_dim, embed_dim // 2),
            nn.SiLU(),
            nn.Dropout(p=0.2),
            nn.Linear(embed_dim // 2, num_classes)
        )

    def forward(self, x):
        x = self.stage1(x)
        x = self.drop2d(x)
        x = self.ablock(x)
        x, hcoord, wcoord, Hp, Wp = self.patch(x)
        x = x + pos2d_sincos(Hp, Wp, x.size(-1), x.device)
        for blk in self.blocks:
            x = blk(x, hcoord, wcoord)
        x = self.final_norm(x)
        x_mean = x.mean(dim=1)
        x_max = x.max(dim=1).values
        x = x_mean + x_max
        return self.head(x)

# ==============================================================================
# SECTION 3: Training Strategy & Configuration
# ==============================================================================
import json
from sklearn.metrics import log_loss

CFG = dict(
    SEEDS=[42],
    EPOCHS=150,
    PATIENCE=35,
    LR=2e-4,
    WARMUP_EPOCHS=5,
    WEIGHT_DECAY=1e-3,
    EMA_DECAY=0.99,
    DROPOUT_SCALE=0.5,
    CLASS_WEIGHT_MODE="cb",
    CB_BETA=0.999,
    CLASS_WEIGHT_MAX_RATIO=3.0,  # cap max/min class weight so no class can dominate the loss
    # FIX round 2: class 2 (206 train images) is being systematically stolen by
    # class 1 (92 true class-2 test images -> 43 predicted class 1, only 25
    # correct). Class 2 sits "sandwiched" between two classes and the current
    # CB weighting gives it a below-1.0 weight (0.92) purely because it has
    # more raw images than 1 or 4 -- but raw count isn't why it's failing;
    # it's failing because it's visually adjacent to both neighbors. A small
    # explicit boost on top of the CB weight targets that directly. Set to
    # 1.0 to disable.
    CLASS_WEIGHT_OVERRIDE={2: 1.25},
    FOCAL_GAMMA=1.5,
    # FIX round 2: ordinal label smoothing spreads some probability mass onto
    # neighboring grades by design (that's what makes it "ordinal-aware"), but
    # combined with a strong class-1 weight this was likely adding to the
    # 1<->2 bleed. Trimming the smoothing amount keeps the ordinal awareness
    # but weakens the bleed.
    LABEL_SMOOTH=0.07,
    ORDINAL_SMOOTH=True,
    SCHEDULE=[(20, 0.00, 0.30), (75, 0.15, 0.40), (10**9, 0.25, 0.50)],
    MIX_ALPHA=0.4,
    TTA_MODES=["none", "hflip", "flip", "d4"],
    # FIX: RetinaMNIST's val split is tiny (~120 images), so the raw per-epoch
    # score is noisy -- a smoothing window of 3 still lets a lucky/unlucky
    # epoch flip which checkpoint gets saved as "best". Widening it to 5
    # trades a bit of responsiveness for a steadier, more reproducible pick.
    SELECT_SMOOTH=5,
    EVAL_TEST=True,
    # FIX: QWK is an ordinal metric -- it forgives "off by one grade" mistakes
    # almost completely, so a checkpoint can hedge toward the crowded middle
    # classes (1/2/3), rack up a good QWK/accuracy, and still be genuinely bad
    # per-class (low macro F1, low BalAcc, a messy confusion matrix). Weighting
    # it at 0.30 was letting checkpoint selection pick exactly that kind of
    # model. Lowering it makes F1/BalAcc (which is what "the confusion matrix
    # looks bad" is actually measuring) drive checkpoint selection instead.
    QWK_WEIGHT=0.12,
    SWA_START_FRAC=0.65,
    DECOUPLE_EPOCHS=10,
    DECOUPLE_LR=5e-5,
)
MODEL_KW = dict(base=64, embed_dim=256, depth=4)

def build_model():
    model = ImprovedRetinaNet(in_ch=3, num_classes=n_classes, **MODEL_KW).to(DEVICE)
    for m in model.modules():
        if isinstance(m, (nn.Dropout, nn.Dropout2d)):
            m.p = m.p * CFG["DROPOUT_SCALE"]
    return model

def make_class_weights(counts, mode, beta, max_ratio=None, override=None):
    counts = np.asarray(counts, dtype=np.float64)
    if mode == "cb":
        eff = 1.0 - np.power(beta, counts)
        w = (1.0 - beta) / np.maximum(eff, 1e-12)
    elif mode == "sqrt":
        w = 1.0 / np.sqrt(counts)
    else:
        w = np.ones_like(counts)
    if max_ratio is not None and max_ratio > 0:
        # clip the weight spread. Without this, a single very rare class
        # can get a weight several times any other class's, which teaches the
        # model to spam that class's predictions rather than genuinely
        # separate it -- exactly what inflates its recall while collapsing
        # its precision.
        w = np.clip(w, w.min(), w.min() * max_ratio)
    if override:
        # manual per-class multiplier applied *after* the CB/cap logic, for a
        # class that needs a nudge the raw counts don't otherwise justify
        # (see CLASS_WEIGHT_OVERRIDE in CFG).
        for idx, mult in override.items():
            w[idx] = w[idx] * float(mult)
    w = w / w.sum() * len(counts)
    return torch.tensor(w, dtype=torch.float32)

class OrdinalFocalLoss(nn.Module):
    def __init__(self, class_weights, gamma=1.5, smoothing=0.1, ordinal=True):
        super().__init__()
        self.register_buffer("cw", class_weights)
        self.gamma = float(gamma)
        self.smoothing = float(smoothing)
        self.ordinal = bool(ordinal)

    def forward(self, logits, target):
        C = logits.size(1)
        logp = F.log_softmax(logits.float(), dim=1)
        p_true = logp.gather(1, target.view(-1, 1)).exp().squeeze(1)
        with torch.no_grad():
            if self.ordinal:
                grades = torch.arange(C, device=logits.device).view(1, -1)
                off = torch.exp(-(grades - target.view(-1, 1)).abs().float())
            else:
                off = torch.ones(logits.size(0), C, device=logits.device)
            off.scatter_(1, target.view(-1, 1), 0.0)
            off = off / off.sum(dim=1, keepdim=True)
            soft = self.smoothing * off
            soft.scatter_(1, target.view(-1, 1), 1.0 - self.smoothing)
        ce = -(soft * logp).sum(dim=1)
        loss = self.cw[target] * (1.0 - p_true).clamp_min(0.0).pow(self.gamma) * ce
        return loss.mean()

class EMA:
    def __init__(self, model, decay=0.99):
        self.model = model
        self.decay = decay
        self.step = 0
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}
        self.backup = None

    @torch.no_grad()
    def update(self):
        self.step += 1
        d = min(self.decay, (1.0 + self.step) / (10.0 + self.step))
        for k, v in self.model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(d).add_(v.detach(), alpha=1.0 - d)
            else:
                self.shadow[k].copy_(v)

    def apply_shadow(self):
        self.backup = {k: v.detach().clone() for k, v in self.model.state_dict().items()}
        self.model.load_state_dict(self.shadow)

    def restore(self):
        self.model.load_state_dict(self.backup)
        self.backup = None

def rand_bbox(W, H, lam):
    cut_rat = math.sqrt(1.0 - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    return x1, y1, x2, y2

def mix_batch(imgs, y, p=0.3, alpha=0.2):
    if np.random.rand() > p:
        return imgs, y, None, 1.0, "none"
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(imgs.size(0), device=imgs.device)
    if np.random.rand() < 0.5:
        mixed_x = lam * imgs + (1 - lam) * imgs[index, :]
    else:
        _, _, H, W = imgs.size()
        x1, y1, x2, y2 = rand_bbox(W, H, lam)
        mixed_x = imgs.clone()
        mixed_x[:, :, y1:y2, x1:x2] = imgs[index, :, y1:y2, x1:x2]
        lam = 1.0 - ((x2 - x1) * (y2 - y1) / (W * H + 1e-8))
    return mixed_x, y, y[index], lam, "mixed"

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def compute_metrics(targets, probs):
    probs = np.asarray(probs, dtype=np.float64)
    probs = probs / probs.sum(axis=1, keepdims=True)
    preds = probs.argmax(axis=1)
    acc = accuracy_score(targets, preds)
    p, r, f1, _ = precision_recall_fscore_support(targets, preds, average="macro", zero_division=0)
    bal = balanced_accuracy_score(targets, preds)
    try:
        qwk = cohen_kappa_score(targets, preds, weights="quadratic",
                                 labels=list(range(n_classes)))
    except Exception:
        qwk = 0.0
    try:
        auc = roc_auc_score(targets, probs, multi_class="ovr", labels=list(range(n_classes)))
    except Exception:
        auc = float("nan")
    return dict(acc=acc, prec=p, rec=r, f1=f1, balacc=bal, auc=auc, qwk=qwk)

def selection_score(m):
    auc = 0.5 if np.isnan(m["auc"]) else m["auc"]
    qwk = max(m.get("qwk", 0.0), 0.0)
    qw = CFG["QWK_WEIGHT"]
    rest = 1.0 - qw
    # FIX: F1 + BalAcc now carry most of the "rest" weight (0.45 each instead
    # of 0.35), acc/auc get less say -- acc in particular is the metric most
    # inflated by the majority class, so it should not have equal footing
    # with the two metrics that expose minority-class failure.
    return (rest * 0.45) * m["f1"] + (rest * 0.45) * m["balacc"] + (rest * 0.05) * m["acc"] + (rest * 0.05) * auc + qw * qwk

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, probs_all, targets_all = 0.0, [], []
    for imgs, targets in loader:
        y = targets.view(-1).long().to(DEVICE)
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        total_loss += criterion(logits, y).item() * imgs.size(0)
        probs_all.append(logits.float().softmax(dim=1).cpu().numpy())
        targets_all.append(y.cpu().numpy())
    probs = np.concatenate(probs_all)
    targets = np.concatenate(targets_all)
    m = compute_metrics(targets, probs)
    m["loss"] = total_loss / len(loader.dataset)
    return m

def train_model(seed, class_counts, train_loader, val_loader):
    name = f"RetinaNet_s{seed}"
    print(f"\n{'=' * 50}\nTraining {name}\n{'=' * 50}")
    seed_all(seed)
    model = build_model()
    print(f"Params (M): {sum(p.numel() for p in model.parameters()) / 1e6:.2f}")

    cw = make_class_weights(class_counts, CFG["CLASS_WEIGHT_MODE"], CFG["CB_BETA"], CFG["CLASS_WEIGHT_MAX_RATIO"], CFG["CLASS_WEIGHT_OVERRIDE"])
    print("Class weights:", [round(float(x), 3) for x in cw])
    criterion = OrdinalFocalLoss(cw, CFG["FOCAL_GAMMA"], CFG["LABEL_SMOOTH"], CFG["ORDINAL_SMOOTH"]).to(DEVICE)

    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        nd = n.endswith(".bias") or any(t in n.lower() for t in ["bn", "norm"])
        (no_decay if nd else decay).append(p)
    optimizer = torch.optim.AdamW(
        [{"params": decay, "weight_decay": CFG["WEIGHT_DECAY"]},
         {"params": no_decay, "weight_decay": 0.0}], lr=CFG["LR"])

    from torch.optim.lr_scheduler import SequentialLR, LambdaLR, CosineAnnealingLR
    W, E = CFG["WARMUP_EPOCHS"], CFG["EPOCHS"]
    warmup = LambdaLR(optimizer, lr_lambda=lambda e: (e + 1) / W if e < W else 1.0)
    cosine = CosineAnnealingLR(optimizer, T_max=E - W, eta_min=1e-6)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[W])
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))
    ema = EMA(model, decay=CFG["EMA_DECAY"])

    swa_start_epoch = max(1, int(E * CFG["SWA_START_FRAC"]))
    swa_model = AveragedModel(model)
    swa_updates = 0

    best_score, best_epoch, patience_cnt = -float("inf"), 0, 0
    recent_scores = []
    best_path = f"best_{name}.pt"
    best_val = None

    for epoch in range(1, E + 1):
        t0 = time.time()
        for last_ep, mp, sp in CFG["SCHEDULE"]:
            if epoch <= last_ep:
                cur_mix_prob, cur_strong_p = mp, sp
                break
        train_loader.dataset.strong_p = cur_strong_p

        model.train()
        tr_loss, n_seen = 0.0, 0
        tr_preds, tr_tgts = [], []
        for imgs, targets in train_loader:
            y = targets.view(-1).long().to(DEVICE)
            imgs = imgs.to(DEVICE)
            imgs, y_a, y_b, lam, tag = mix_batch(imgs, y, p=cur_mix_prob, alpha=CFG["MIX_ALPHA"])
            with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
                logits = model(imgs)
            if tag == "mixed":
                loss = mixup_criterion(criterion, logits, y_a, y_b, lam)
            else:
                loss = criterion(logits, y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=3.0)
            scaler.step(optimizer)
            scaler.update()
            ema.update()

            tr_loss += loss.item() * imgs.size(0)
            n_seen += imgs.size(0)
            tr_preds.extend(logits.argmax(1).detach().cpu().tolist())
            tr_tgts.extend((y_a if tag == "mixed" else y).cpu().tolist())
        scheduler.step()

        tr_acc = accuracy_score(tr_tgts, tr_preds)
        tr_p, tr_r, tr_f1, _ = precision_recall_fscore_support(tr_tgts, tr_preds, average="macro", zero_division=0)

        ema.apply_shadow()

        if epoch >= swa_start_epoch:
            swa_model.update_parameters(model)
            swa_updates += 1

        va = evaluate(model, val_loader, criterion)
        recent_scores.append(selection_score(va))
        score = float(np.mean(recent_scores[-CFG["SELECT_SMOOTH"]:]))
        line = (f"[Epoch {epoch:02d}/{E}] Train: loss={tr_loss / n_seen:.4f} acc={tr_acc:.3f} "
                f"P={tr_p:.3f} R={tr_r:.3f} F1={tr_f1:.3f} | "
                f"Val: loss={va['loss']:.4f} acc={va['acc']:.3f} P={va['prec']:.3f} R={va['rec']:.3f} "
                f"F1={va['f1']:.3f} BalAcc={va['balacc']:.3f} QWK={va['qwk']:.3f} AUC={va['auc']:.3f} | "
                f"time={time.time() - t0:.1f}s lr={scheduler.get_last_lr()[0]:.2e}")
        stop = False
        if score > best_score + 1e-6:
            best_score, best_epoch, patience_cnt, best_val = score, epoch, 0, va
            torch.save(model.state_dict(), best_path)
            print(line + " --> Best Model Saved!")
        else:
            patience_cnt += 1
            print(line)
            if patience_cnt >= CFG["PATIENCE"]:
                print(f"Early stopping at epoch {epoch} (best epoch {best_epoch}, score={best_score:.3f})")
                stop = True
        ema.restore()
        if stop:
            break

    swa_path, swa_score, swa_val = None, -float("inf"), None
    if swa_updates > 0:
        update_bn(train_loader, swa_model, device=DEVICE)
        swa_val = evaluate(swa_model.module, val_loader, criterion)
        swa_score = selection_score(swa_val)
        swa_path = f"swa_{name}.pt"
        torch.save(swa_model.module.state_dict(), swa_path)
        print(f"[SWA] updates={swa_updates} | Val: acc={swa_val['acc']:.3f} F1={swa_val['f1']:.3f} "
              f"BalAcc={swa_val['balacc']:.3f} QWK={swa_val['qwk']:.3f} AUC={swa_val['auc']:.3f} "
              f"| selection_score={swa_score:.3f} (EMA-best was {best_score:.3f})")

    if swa_score > best_score:
        print("[SELECT] SWA checkpoint beats EMA-best checkpoint on validation -> using SWA as final backbone.")
        final_path, final_score, final_val = swa_path, swa_score, swa_val
        chosen = "swa"
    else:
        print("[SELECT] EMA-best checkpoint kept as final backbone.")
        final_path, final_score, final_val = best_path, best_score, best_val
        chosen = "ema_best"

    return final_path, final_score, best_epoch, final_val, chosen

def make_balanced_sampler(labels, num_classes):
    counts = np.bincount(labels, minlength=num_classes).astype(np.float64)
    class_w = 1.0 / np.maximum(counts, 1.0)
    sample_w = class_w[labels]
    return torch.utils.data.WeightedRandomSampler(
        weights=torch.tensor(sample_w, dtype=torch.double),
        num_samples=len(sample_w),
        replacement=True
    )

def decoupled_head_finetune(model, train_loader, labels_train, val_loader, criterion,
                             epochs=10, lr=5e-5):
    print(f"\n{'=' * 50}\nStage 2: decoupled classifier-head fine-tuning\n{'=' * 50}")
    base_val = evaluate(model, val_loader, criterion)
    base_score = selection_score(base_val)
    print(f"[DECOUPLE] starting score={base_score:.3f} (acc={base_val['acc']:.3f} F1={base_val['f1']:.3f} "
          f"BalAcc={base_val['balacc']:.3f} QWK={base_val['qwk']:.3f})")

    for p in model.parameters():
        p.requires_grad = False
    for p in model.head.parameters():
        p.requires_grad = True

    sampler = make_balanced_sampler(labels_train, n_classes)
    balanced_loader = DataLoader(
        train_loader.dataset, batch_size=BATCH_SIZE, sampler=sampler,
        num_workers=0, pin_memory=True, drop_last=False
    )
    opt = torch.optim.AdamW(model.head.parameters(), lr=lr, weight_decay=1e-4)

    best_state = copy.deepcopy(model.state_dict())
    best_score = base_score
    best_val = base_val

    for ep in range(1, epochs + 1):
        model.train()
        for imgs, y in balanced_loader:
            imgs = imgs.to(DEVICE)
            y = y.view(-1).long().to(DEVICE)
            logits = model(imgs)
            loss = criterion(logits, y)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        va = evaluate(model, val_loader, criterion)
        sc = selection_score(va)
        print(f"[DECOUPLE ep {ep:02d}/{epochs}] Val: acc={va['acc']:.3f} F1={va['f1']:.3f} "
              f"BalAcc={va['balacc']:.3f} QWK={va['qwk']:.3f} AUC={va['auc']:.3f} | score={sc:.3f}")
        if sc > best_score:
            best_score, best_val = sc, va
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    for p in model.parameters():
        p.requires_grad = True

    if best_score > base_score:
        print(f"[DECOUPLE] kept fine-tuned head: {base_score:.3f} -> {best_score:.3f}")
    else:
        print(f"[DECOUPLE] fine-tuning did not beat stage-1 model ({base_score:.3f}); reverted to stage-1 weights.")
    return model, best_val, (best_score > base_score)

def _views(imgs, mode):
    if mode == "none":
        return [imgs]
    if mode == "hflip":
        return [imgs, torch.flip(imgs, [3])]
    v = [imgs, torch.flip(imgs, [3]), torch.flip(imgs, [2]), torch.flip(imgs, [2, 3])]
    if mode == "flip":
        return v
    hf = torch.flip(imgs, [3])
    v += [torch.rot90(imgs, 1, [2, 3]), torch.rot90(imgs, 3, [2, 3]),
          torch.rot90(hf, 1, [2, 3]), torch.rot90(hf, 3, [2, 3])]
    return v

@torch.no_grad()
def predict_probs(model, loader, tta_mode):
    model.eval()
    ps, ts = [], []
    for imgs, targets in loader:
        imgs = imgs.to(DEVICE)
        p = torch.stack([model(v).float().softmax(dim=1) for v in _views(imgs, tta_mode)]).mean(0)
        ps.append(p.cpu().numpy())
        ts.append(targets.view(-1).numpy())
    probs = np.concatenate(ps).astype(np.float64)
    probs = probs / probs.sum(axis=1, keepdims=True)
    return probs, np.concatenate(ts)

def print_report(title, targets, probs):
    preds = probs.argmax(axis=1)
    m = compute_metrics(targets, probs)
    try:
        ll = log_loss(targets, probs, labels=list(range(n_classes)))
    except Exception:
        ll = float("nan")
    print(f"\n[{title}]")
    print(f"loss={ll:.4f} | acc={m['acc']:.3f} | P={m['prec']:.3f} | R={m['rec']:.3f} | "
          f"F1={m['f1']:.3f} | BalAcc={m['balacc']:.3f} | QWK={m['qwk']:.3f} | AUC(ovr)={m['auc']:.3f}")
    print(classification_report(targets, preds, labels=list(range(n_classes)), digits=3, zero_division=0))
    print("Confusion matrix (rows=true, cols=pred):")
    print(confusion_matrix(targets, preds, labels=list(range(n_classes))))
    return m

def main():
    print("CONFIG:", json.dumps({k: v for k, v in CFG.items()}, default=str))
    train_loader, val_loader, test_loader, class_counts, labels_train = get_dataloaders()
    print("Train class counts:", class_counts.tolist())

    cw = make_class_weights(class_counts, CFG["CLASS_WEIGHT_MODE"], CFG["CB_BETA"], CFG["CLASS_WEIGHT_MAX_RATIO"], CFG["CLASS_WEIGHT_OVERRIDE"])
    criterion = OrdinalFocalLoss(cw, CFG["FOCAL_GAMMA"], CFG["LABEL_SMOOTH"], CFG["ORDINAL_SMOOTH"]).to(DEVICE)

    runs = []
    for seed in CFG["SEEDS"]:
        path, score, ep, val_m, chosen = train_model(seed, class_counts, train_loader, val_loader)
        runs.append(dict(seed=seed, path=path, val_score=float(score), best_epoch=ep, stage1_checkpoint=chosen))

    models = []
    for r in runs:
        m = build_model()
        m.load_state_dict(torch.load(r["path"], map_location=DEVICE))
        m.eval()
        models.append(m)
    model = models[0]

    model, val_after_decouple, decouple_kept = decoupled_head_finetune(
        model, train_loader, labels_train, val_loader, criterion,
        epochs=CFG["DECOUPLE_EPOCHS"], lr=CFG["DECOUPLE_LR"]
    )
    runs[0]["decouple_kept"] = bool(decouple_kept)
    models = [model]

    best_mode, best_ll, val_probs, val_t = None, float("inf"), None, None
    for mode in CFG["TTA_MODES"]:
        vps = []
        for m in models:
            vp, vt = predict_probs(m, val_loader, mode)
            vps.append(vp)
        ens = np.mean(vps, axis=0)
        ll = log_loss(vt, ens, labels=list(range(n_classes)))
        vm = compute_metrics(vt, ens)
        print(f"[VAL] TTA={mode:6s} logloss={ll:.4f} acc={vm['acc']:.3f} F1={vm['f1']:.3f} "
              f"BalAcc={vm['balacc']:.3f} QWK={vm['qwk']:.3f} AUC={vm['auc']:.3f}")
        if ll < best_ll:
            best_mode, best_ll, val_probs, val_t = mode, ll, ens, vt
    print(f"--> TTA mode chosen on val: {best_mode}")

    val_m = compute_metrics(val_t, val_probs)
    print(f"[VAL] final: acc={val_m['acc']:.3f} P={val_m['prec']:.3f} R={val_m['rec']:.3f} "
          f"F1={val_m['f1']:.3f} BalAcc={val_m['balacc']:.3f} QWK={val_m['qwk']:.3f} AUC={val_m['auc']:.3f}")

    if not CFG["EVAL_TEST"]:
        print_report("VAL - final pipeline", val_t, val_probs)
        with open("experiment_record.json", "w") as f:
            json.dump(dict(config=CFG, model_kw=MODEL_KW, runs=runs, tta_mode=best_mode,
                           val={k: float(v) for k, v in val_m.items()}), f, indent=2, default=str)
        print("EVAL_TEST=False -> test set NOT evaluated. Saved experiment_record.json")
        return

    tps = []
    for m in models:
        tp, tt = predict_probs(m, test_loader, best_mode)
        tps.append(tp)
    test_probs = np.mean(tps, axis=0)
    test_m = print_report("FINAL TEST", tt, test_probs)

    record = dict(
        config=CFG, model_kw=MODEL_KW, img_size=IMG_SIZE, batch_size=BATCH_SIZE,
        runs=runs, tta_mode=best_mode,
        val={k: float(v) for k, v in val_m.items()},
        test={k: float(v) for k, v in test_m.items()},
    )
    with open("experiment_record.json", "w") as f:
        json.dump(record, f, indent=2, default=str)
    print("Saved experiment_record.json")

if __name__ == "__main__":
    main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 2.9 MB/s eta 0:00:00
Using Device: cuda
CONFIG: {"SEEDS": [42], "EPOCHS": 150, "PATIENCE": 35, "LR": 0.0002, "WARMUP_EPOCHS": 5, "WEIGHT_DECAY": 0.001, "EMA_DECAY": 0.99, "DROPOUT_SCALE": 0.5, "CLASS_WEIGHT_MODE": "cb", "CB_BETA": 0.999, "CLASS_WEIGHT_MAX_RATIO": 3.0, "CLASS_WEIGHT_OVERRIDE": {"2": 1.25}, "FOCAL_GAMMA": 1.5, "LABEL_SMOOTH": 0.07, "ORDINAL_SMOOTH": true, "SCHEDULE": [[20, 0.0, 0.3], [75, 0.15, 0.4], [1000000000, 0.25, 0.5]], "MIX_ALPHA": 0.4, "TTA_MODES": ["none", "hflip", "flip", "d4"], "SELECT_SMOOTH": 5, "EVAL_TEST": true, "QWK_WEIGHT": 0.12, "SWA_START_FRAC": 0.65, "DECOUPLE_EPOCHS": 10, "DECOUPLE_LR": 5e-05}


100%|██████████| 128M/128M [00:56<00:00, 2.25MB/s] 


Train class counts: [486, 128, 206, 194, 66]

Training RetinaNet_s42
Params (M): 7.99
Class weights: [0.425, 1.275, 1.098, 0.927, 1.275]
[Epoch 01/150] Train: loss=0.8920 acc=0.241 P=0.217 R=0.214 F1=0.156 | Val: loss=0.8841 acc=0.450 P=0.090 R=0.200 F1=0.124 BalAcc=0.200 QWK=0.000 AUC=0.564 | time=50.8s lr=8.00e-05 --> Best Model Saved!
[Epoch 02/150] Train: loss=0.8735 acc=0.338 P=0.208 R=0.232 F1=0.191 | Val: loss=0.8386 acc=0.483 P=0.241 R=0.308 F1=0.235 BalAcc=0.308 QWK=0.483 AUC=0.648 | time=16.6s lr=1.20e-04 --> Best Model Saved!
[Epoch 03/150] Train: loss=0.8070 acc=0.435 P=0.278 R=0.291 F1=0.259 | Val: loss=0.7307 acc=0.550 P=0.344 R=0.353 F1=0.307 BalAcc=0.353 QWK=0.494 AUC=0.722 | time=16.3s lr=1.60e-04 --> Best Model Saved!
[Epoch 04/150] Train: loss=0.7832 acc=0.468 P=0.281 R=0.309 F1=0.275 | Val: loss=0.6937 acc=0.575 P=0.338 R=0.361 F1=0.304 BalAcc=0.361 QWK=0.592 AUC=0.749 | time=16.4s lr=2.00e-04 --> Best Model Saved!
[Epoch 05/150] Train: loss=0.7836 acc=0.460 P=0.300